# k-means clustering

Unsupervised learning has no labels to predict. The question is what structure the data has by itself. k-means is the simplest answer: split the points into $k$ groups so that each point is close to the centre of its own group. It is fast, easy to explain and a good default, and it has strong assumptions that are worth knowing before you trust a result.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

from common.datasets import make_blobs_data, make_circles_data, make_customer_data, make_moons_data
from common.plotting import plot_clusters

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

A shop wants to send three kinds of offer and has 1,000 customers. Nobody has labelled them as "bargain hunter" or "big spender". k-means looks only at the numbers and proposes groups.

Picture three flags planted in a field of points. Each point walks to its nearest flag. Then each flag moves to the middle of the points that walked to it. Repeat until nothing moves. The flags are the **centroids**, and each group is a **cluster**.

You choose $k$ before you start. The algorithm never tells you whether $k$ was a good choice, which is the main practical problem (section 6).

## 2. Maths and cost function

Given points $\mathbf x_1,\dots,\mathbf x_n$ and $k$ centroids $\boldsymbol\mu_1,\dots,\boldsymbol\mu_k$, k-means minimises the **inertia**, the total squared distance from each point to its assigned centroid:

$$J=\sum_{i=1}^{n}\lVert \mathbf x_i-\boldsymbol\mu_{c(i)}\rVert^2,\qquad c(i)=\arg\min_j \lVert\mathbf x_i-\boldsymbol\mu_j\rVert^2$$

Searching all possible assignments is hopeless, so **Lloyd's algorithm** alternates two steps:

1. **Assign.** Fix the centroids, give each point to its nearest one. This is the best assignment for those centroids.
2. **Update.** Fix the assignments, move each centroid to the mean of its points. The mean is the point that minimises the sum of squared distances to a set (set the derivative of $\sum\lVert\mathbf x_i-\boldsymbol\mu\rVert^2$ to zero), so this step cannot raise $J$.

Both steps lower $J$ or leave it unchanged, and there are finitely many assignments, so the loop always stops. It stops at a **local** minimum, not necessarily the best one. Where the centroids start matters.

Three things follow from the squared Euclidean cost. Clusters are pulled towards round blobs of similar size. Features with a large scale dominate. A single far-away point drags a centroid, because its squared distance is huge.

## 3. From scratch

Start with the data: 300 points from three blobs, with the true blob kept aside only for checking.

In [ ]:
X_df, y_true = make_blobs_data(n_samples=300, centers=3, cluster_std=1.0, seed=SEED)
X = X_df.to_numpy()


def kmeans_plus_plus(X: np.ndarray, k: int, rng: np.random.Generator) -> np.ndarray:
    """Pick k starting centroids: the first at random, each next one far from those chosen."""
    centers = [X[rng.integers(len(X))]]
    for _ in range(k - 1):
        d2 = np.min([((X - c) ** 2).sum(axis=1) for c in centers], axis=0)
        centers.append(X[rng.choice(len(X), p=d2 / d2.sum())])  # far points are likelier
    return np.array(centers)


def lloyd(X: np.ndarray, centers: np.ndarray, max_iter: int = 100):
    """Run assign/update until the centroids stop moving. Returns labels, centres, inertia, history."""
    history = []
    for _ in range(max_iter):
        d2 = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)  # (n, k)
        labels = d2.argmin(axis=1)
        history.append(d2[np.arange(len(X)), labels].sum())
        new = np.array(
            [
                X[labels == j].mean(axis=0) if (labels == j).any() else centers[j]
                for j in range(len(centers))
            ]
        )
        if np.allclose(new, centers):
            break
        centers = new
    return labels, centers, history[-1], history


init = X[rng.choice(len(X), 3, replace=False)]  # plain random start
labels, centers, inertia, history = lloyd(X, init)
print("iterations:", len(history), " inertia per iteration:", np.round(history, 1))

Inertia drops every iteration and then stops changing, as the argument above promised. Here is the first iterations as pictures, starting from a deliberately poor start (all three centroids in one corner).

In [ ]:
bad_init = X[np.argsort(X[:, 0])[:3]]  # the three leftmost points
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), sharex=True, sharey=True)
c = bad_init
for ax, step in zip(axes, range(4)):
    d2 = ((X[:, None, :] - c[None]) ** 2).sum(axis=2)
    lab = d2.argmin(axis=1)
    plot_clusters(
        X, lab, ax, centers=c, title=f"after {step} updates: J={d2.min(axis=1).sum():.0f}"
    )
    c = np.array([X[lab == j].mean(axis=0) if (lab == j).any() else c[j] for j in range(3)])
plt.show()

## 4. Initialisation and k-means++

Because Lloyd's algorithm only finds a local minimum, a bad start can leave two centroids sharing one blob while another blob has to share. Run it from 200 random starts and look at where the inertia ends up. Use four blobs, which makes bad outcomes easier to produce.

In [ ]:
X4_df, _ = make_blobs_data(n_samples=400, centers=4, cluster_std=0.9, seed=3)
X4 = X4_df.to_numpy()
rand_J, pp_J = [], []
for s in range(200):
    r = np.random.default_rng(s)
    rand_J.append(lloyd(X4, X4[r.choice(len(X4), 4, replace=False)])[2])
    pp_J.append(lloyd(X4, kmeans_plus_plus(X4, 4, r))[2])
print(
    f"random start:  best {min(rand_J):.0f}, worst {max(rand_J):.0f}, runs stuck above best: {np.mean(np.array(rand_J) > min(rand_J) * 1.05):.0%}"
)
print(
    f"k-means++:     best {min(pp_J):.0f}, worst {max(pp_J):.0f}, runs stuck above best: {np.mean(np.array(pp_J) > min(pp_J) * 1.05):.0%}"
)

**k-means++** picks the first centroid at random and then picks each later one with probability proportional to its squared distance from the nearest centroid already chosen. Starts are spread out, so fewer runs land in a poor local minimum. It is also provably within a factor of $O(\log k)$ of the best possible inertia on average. The cheap, standard defence on top of that is to run several starts and keep the lowest inertia, which is what `n_init` does.

## 5. Library version

`KMeans` does the same with k-means++ by default and several restarts. Check it agrees with the from-scratch code when given the same starting centroids.

In [ ]:
start = kmeans_plus_plus(X, 3, np.random.default_rng(1))
mine = lloyd(X, start)
ref = KMeans(3, init=start, n_init=1, algorithm="lloyd").fit(X)
print("inertia, mine vs sklearn:", round(mine[2], 3), round(ref.inertia_, 3))
print("same labels:", np.array_equal(mine[0], ref.labels_))

km = KMeans(3, n_init=10, random_state=SEED).fit(X)
print("agreement with the true blobs (ARI):", round(adjusted_rand_score(y_true, km.labels_), 3))
fig, ax = plt.subplots()
plot_clusters(X, km.labels_, ax, centers=km.cluster_centers_, title="KMeans(3) on three blobs")
plt.show()

`adjusted_rand_score` (ARI) compares two groupings without caring what the groups are called. 1 is identical, around 0 is what random labelling gives. It needs true labels, so it is a teaching tool here. Notebook 06 covers what to use when you have no labels.

## 6. Choosing k

There is no labelled answer, so we look for a $k$ that the data supports. Two common tools.

**Elbow.** Inertia always falls as $k$ grows (with $k=n$ it is zero), so we cannot pick the minimum. We look for the bend where extra clusters stop buying much.

**Silhouette.** For each point, $a$ is its mean distance to the other points in its own cluster and $b$ is its mean distance to the points in the nearest other cluster. The silhouette is $s=\frac{b-a}{\max(a,b)}$. It is near 1 when a point sits well inside its cluster, near 0 on a border and negative if it looks closer to another cluster. The mean over all points is a single score you can compare across $k$.

In [ ]:
ks = range(2, 10)
inertias, sils = [], []
for k in ks:
    m = KMeans(k, n_init=10, random_state=SEED).fit(X4)
    inertias.append(m.inertia_)
    sils.append(silhouette_score(X4, m.labels_))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].plot(list(ks), inertias, marker="o")
axes[0].set(xlabel="k", ylabel="inertia", title="elbow")
axes[1].plot(list(ks), sils, marker="o")
axes[1].set(xlabel="k", ylabel="mean silhouette", title="silhouette")
plt.show()
print("best k by silhouette:", list(ks)[int(np.argmax(sils))])

The four-blob data bends sharply at $k=4$ and the silhouette peaks there too. Real data is rarely this clean. Try the customer data, which has four hidden segments. The features are standardised first (section 7 shows why).

In [ ]:
Xc, seg = make_customer_data(n_samples=1000, seed=SEED)
Xc_s = StandardScaler().fit_transform(Xc)
print(Xc.describe().loc[["mean", "std"]].round(2))
ks = range(2, 10)
res = []
for k in ks:
    m = KMeans(k, n_init=10, random_state=SEED).fit(Xc_s)
    res.append((k, m.inertia_, silhouette_score(Xc_s, m.labels_)))
print(pd.DataFrame(res, columns=["k", "inertia", "silhouette"]).round(3).to_string(index=False))

Here the silhouette peaks at $k=4$ (0.55, against 0.45 at $k=3$ and $k=5$) and inertia falls by about half from 3 to 4 but only by 11% from 4 to 5. The data were generated with four segments, so that is the right answer, and it is a clean case. Real data often gives a soft elbow and a flat silhouette, and then either of two neighbouring values is defensible. Business use decides: how many segments can the marketing team act on? Treat the choice of $k$ as a judgement supported by these numbers, not a result.

## 7. Scaling

k-means uses raw Euclidean distance, so a feature in dollars drowns one measured as a share. In the customer data `annual_spend` runs into the thousands and `discount_share` lives between 0 and 1.

In [ ]:
raw = KMeans(4, n_init=10, random_state=SEED).fit(Xc)
scaled = KMeans(4, n_init=10, random_state=SEED).fit(Xc_s)
print("ARI vs true segments, unscaled:", round(adjusted_rand_score(seg, raw.labels_), 3))
print("ARI vs true segments, scaled:  ", round(adjusted_rand_score(seg, scaled.labels_), 3))
print(
    "share of squared distance from annual_spend alone (unscaled):",
    round(float(Xc["annual_spend"].var() / Xc.var().sum()), 3),
)

Unscaled, essentially all of the distance comes from spend and the other four columns do not count, and agreement with the true segments drops from 0.99 to 0.57. Standardise first (zero mean, unit variance). Do it inside a pipeline if you later score on new data, so the means and variances come from training data only.

## 8. What breaks

k-means assumes round, similarly sized clusters of similar density, and no extreme outliers. Break each in turn.

**Non-spherical clusters.** Two moons and two rings each have an obvious two-group structure, and no straight line between centroids separates them.

In [ ]:
Xm_df, ym = make_moons_data(300, noise=0.08, seed=SEED)
Xr_df, yr = make_circles_data(300, noise=0.05, seed=SEED)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (Xs, ys, name) in zip(
    axes, [(Xm_df.to_numpy(), ym, "moons"), (Xr_df.to_numpy(), yr, "circles")]
):
    m = KMeans(2, n_init=10, random_state=SEED).fit(Xs)
    plot_clusters(
        Xs,
        m.labels_,
        ax,
        centers=m.cluster_centers_,
        title=f"{name}: ARI {adjusted_rand_score(ys, m.labels_):.2f}",
    )
plt.show()

ARI is close to zero on both: k-means cuts each shape in half. Notebook 02 shows methods that follow the shape instead.

**Unequal sizes and spreads.** The cost adds squared distances, so a big loose cluster contributes far more to inertia than a small tight one. The cheapest way to cut inertia is often to split the big cluster and let the small one ride along.

In [ ]:
r = np.random.default_rng(SEED)
big = r.normal([0, 0], 2.5, size=(500, 2))
small = r.normal([3, 0], 0.5, size=(40, 2))
Xu = np.vstack([big, small])
yu = np.r_[np.zeros(500), np.ones(40)]
m = KMeans(2, n_init=10, random_state=SEED).fit(Xu)
fig, ax = plt.subplots()
plot_clusters(
    Xu,
    m.labels_,
    ax,
    centers=m.cluster_centers_,
    title=f"unequal clusters: ARI {adjusted_rand_score(yu, m.labels_):.2f}",
)
plt.show()
print("cluster sizes:", np.bincount(m.labels_), " true sizes: [500 40]")

k-means returns two clusters of roughly 290 and 250 points. It cut the big cluster in half, and the 40 tight points are absorbed into one of the halves. ARI is about zero. The boundary sits halfway between centroids, not where the density dips.

**Outliers.** A few extreme points pull a centroid, or get a centroid of their own at the cost of splitting a real cluster.

In [ ]:
Xo = np.vstack([X, [[60, 60], [62, 58], [58, 62]]])
m_clean = KMeans(3, n_init=10, random_state=SEED).fit(X)
m_out = KMeans(3, n_init=10, random_state=SEED).fit(Xo)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharex=True, sharey=True)
plot_clusters(X, m_clean.labels_, axes[0], m_clean.cluster_centers_, "clean data")
plot_clusters(Xo, m_out.labels_, axes[1], m_out.cluster_centers_, "three extreme points added")
plt.show()
print("cluster sizes with outliers:", np.bincount(m_out.labels_))

Three points out of 303 take a whole cluster to themselves (sizes 100, 200 and 3), and two real blobs are merged into one cluster of 200. Remove outliers first, use a robust variant (k-medoids), or use a method that has a noise label (DBSCAN, notebook 02).

**Other limits.** The result depends on the seed unless you use several starts. Hard assignment hides uncertainty for points on a border (notebook 03 fixes this). In very high dimensions Euclidean distances flatten out (see `03_classical_ml/21_knn.ipynb`), so reduce dimensions first (notebook 04).

**Cost.** Each iteration is $O(nkd)$, linear in the data, which is why k-means scales to millions of rows. `MiniBatchKMeans` trades a little quality for even more speed.

Next, methods that do not need $k$ up front and do not assume round clusters.